# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shaheerkhan1117/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

**Logistic regression**, on the same "honest" feature set from `w03_feature_leakage_check`
(`total_impressions`, `total_clicks`, `avg_position`, `active_days`, `position_volatility` +
its fill flag, `ctr`, `impressions_per_active_day`, `ga4_*` one-hot) — deliberately **without**
`pct_change`, the column that defines `is_declining` in the first place.

**Why this fits the lane, not just "why logistic regression":** the whole point of the Refresh
/ Content Opportunity Scoring lane is a ranked queue a content team can act on and explain. A
model whose coefficients map straight onto reason codes (this feature pushed the score up, that
one pulled it down) is worth more here than a few extra AUC points from an ensemble no one on
that team can question. Regularized logistic regression keeps that property; a gradient-boosted
tree was considered and set aside for exactly that reason, not because it would score worse.

**Why leave `pct_change` out even though the baseline rule uses it:** `w03_data_contract`
already showed what happens when a label-derived column gets used as a feature — the score hits
1.000 and means nothing. `is_declining` is *defined* from `pct_change`; a model that gets to see
`pct_change` isn't predicting decline, it's being handed the answer. Section 3 below makes that
comparison explicit instead of hiding it.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/shaheerkhan1117/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "huggingface_hub"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import getpass
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

import duckdb, numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLES = {
    "dim_clients":    f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content":    f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily":     f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}
MONTH = "2026-03"  # same mid-panel month used across w03/w04 this cycle

raw = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions)                                            AS total_impressions,
        SUM(gsc_clicks)                                                 AS total_clicks,
        AVG(gsc_avg_position)                                           AS avg_position,
        COUNT(DISTINCT report_date) FILTER (WHERE gsc_impressions > 0)  AS active_days,
        STDDEV_SAMP(gsc_avg_position)                                   AS position_volatility,
        MODE(ga4_data_available)                                       AS ga4_data_available,
        SUM(CASE WHEN report_date < DATE '{MONTH}-16' THEN gsc_impressions ELSE 0 END) AS imp_first_half,
        SUM(CASE WHEN report_date >= DATE '{MONTH}-16' THEN gsc_impressions ELSE 0 END) AS imp_second_half
    FROM {TABLES['fact_daily']}
    WHERE month = '{MONTH}'
    GROUP BY 1, 2
    HAVING SUM(CASE WHEN report_date < DATE '{MONTH}-16' THEN gsc_impressions ELSE 0 END) > 0
""").df()

df = raw.copy()
df["ctr"] = (df["total_clicks"] / df["total_impressions"]).round(4)
df["impressions_per_active_day"] = (df["total_impressions"] / df["active_days"]).round(2)
df["volatility_is_filled"] = df["position_volatility"].isna().astype(int)
df["position_volatility"] = df["position_volatility"].fillna(0.0)
ga4_dummies = pd.get_dummies(df["ga4_data_available"], prefix="ga4", dummy_na=True)
df = pd.concat([df, ga4_dummies], axis=1)

# The label — identical definition to w03_data_contract / w04_baseline_score, kept separate
# from the honest feature columns below.
df["pct_change"] = (df["imp_second_half"] - df["imp_first_half"]) / df["imp_first_half"]
df["is_declining"] = (df["pct_change"] < -0.2).astype(int)

HONEST_FEATURES = (["total_impressions", "total_clicks", "avg_position", "active_days",
                     "position_volatility", "volatility_is_filled", "ctr",
                     "impressions_per_active_day"] + list(ga4_dummies.columns))

print(f"{len(df):,} rows, {len(HONEST_FEATURES)} honest features, "
      f"{df['is_declining'].mean():.1%} positive rate")

Paste your Hugging Face READ token (hf_...): ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

151,981 rows, 11 honest features, 32.7% positive rate


## 2. Split design

**Grouped by client, not random.** Pages from the same client share client-level baseline
behavior — industry, existing SEO health, how aggressively they publish — that a random
per-page split would let leak across train and test: the model could partly learn "this is one
of Client X's pages" rather than a signal that generalizes to a client it's never seen. A
`GroupShuffleSplit` on `client_hash_id` keeps every client's pages entirely on one side.

**Not time-aware, and that's a named limitation, not an oversight.** `is_declining` is defined
within this single month (front half vs. back half), so there's no earlier month to train on
and later month to test against without pulling more data than this notebook does. The panel's
real final month (`2026-06`) stays sealed per the `w03_data_contract` warning — it's for a true
held-out check later, not for iterating here. A grouped split is the honest choice available
right now; a time-aware one would be the honest choice for an actual forecast.

In [2]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df["client_hash_id"]))
train_df, test_df = df.iloc[train_idx], df.iloc[test_idx]

overlap = set(train_df["client_hash_id"]) & set(test_df["client_hash_id"])
print(f"Train: {len(train_df):,} rows, {train_df['client_hash_id'].nunique()} clients")
print(f"Test:  {len(test_df):,} rows, {test_df['client_hash_id'].nunique()} clients")
print(f"Clients present in both train and test: {len(overlap)} (must be 0 for a clean grouped split)")
assert len(overlap) == 0, "client leaked across the split"

Train: 110,403 rows, 30 clients
Test:  41,578 rows, 14 clients
Clients present in both train and test: 0 (must be 0 for a clean grouped split)


## 3. Train + compare vs my baseline

Three rows, same test set, same metric (AUC), so they're actually comparable:

1. **This model** — logistic regression on the honest features, fit on the grouped train split,
   scored on the held-out grouped test split.
2. **`w03_data_contract`'s quick demo, for reference** — the same honest 5-core-feature logistic
   regression from that notebook, but fit on a random (not grouped) split. Its real captured
   result was AUC 0.610. Repeating that number here rather than re-deriving it, so the two
   numbers sit side by side.
3. **The baseline rule's own `DECLINE_20PLUS` flag** — scored against `is_declining` on this
   notebook's same test set, included **only** to show why it isn't a real prediction: the flag
   *is* `pct_change` past a threshold, and `is_declining` *is* `pct_change` past the same
   threshold, so this row measures agreement with itself, not generalization.

In [3]:
X_train, y_train = train_df[HONEST_FEATURES], train_df["is_declining"]
X_test, y_test = test_df[HONEST_FEATURES], test_df["is_declining"]

scaler = StandardScaler().fit(X_train)
clf = LogisticRegression(max_iter=1000).fit(scaler.transform(X_train), y_train)
model_auc = roc_auc_score(y_test, clf.predict_proba(scaler.transform(X_test))[:, 1])

# Reference row 3 — the baseline rule's flag scored against the label it was built from.
baseline_flag_auc = roc_auc_score(test_df["is_declining"], (test_df["pct_change"] <= -0.20).astype(int))

comparison = pd.DataFrame([
    {"method": "This model — logistic regression, honest features, client-grouped split",
     "split": "grouped by client", "auc": round(model_auc, 3)},
    {"method": "w03_data_contract quick demo — logistic regression, 5 honest features (reference)",
     "split": "random (not grouped)", "auc": 0.610},
    {"method": "Baseline rule's DECLINE_20PLUS flag vs is_declining (definitional, not a prediction)",
     "split": "n/a", "auc": round(baseline_flag_auc, 3)},
])
comparison

,method,split,auc
0,"This model — logistic regression, honest featu...",grouped by client,0.603
1,w03_data_contract quick demo — logistic regres...,random (not grouped),0.610
2,Baseline rule's DECLINE_20PLUS flag vs is_decl...,n/a,0.999


## 4. Errors and interpretation

Two things worth more than the AUC table above: which features the model actually leans on
(logistic regression coefficients, on standardized inputs so they're comparable), and what the
missed cases look like — false negatives (declining pages the model scored low) and false
positives (stable/growing pages it scored high) at a 0.5 cutoff on the held-out test set.

In [4]:
coefs = pd.Series(clf.coef_[0], index=HONEST_FEATURES).sort_values(key=abs, ascending=False)
print("Coefficients (standardized features — larger magnitude = more weight):")
print(coefs.round(3))

preds = clf.predict(scaler.transform(X_test))
print("\nConfusion matrix (rows = actual, cols = predicted), [[TN, FP], [FN, TP]]:")
print(confusion_matrix(y_test, preds))
print()
print(classification_report(y_test, preds, digits=3))

eval_df = test_df.copy()
eval_df["predicted"] = preds
false_negatives = eval_df[(eval_df["is_declining"] == 1) & (eval_df["predicted"] == 0)]
false_positives = eval_df[(eval_df["is_declining"] == 0) & (eval_df["predicted"] == 1)]
correctly_declining = eval_df[(eval_df["is_declining"] == 1) & (eval_df["predicted"] == 1)]

print(f"\nFalse negatives: {len(false_negatives):,}  |  False positives: {len(false_positives):,}")
print("\nMean honest-feature values — missed declines vs. correctly caught declines:")
print(pd.DataFrame({
    "false_negative_mean": false_negatives[HONEST_FEATURES].mean(),
    "correctly_caught_mean": correctly_declining[HONEST_FEATURES].mean(),
}).round(3))

Coefficients (standardized features — larger magnitude = more weight):
impressions_per_active_day   -10.198
total_impressions             10.193
volatility_is_filled           1.947
total_clicks                  -0.664
active_days                   -0.464
ga4_True                       0.126
avg_position                  -0.079
ga4_False                     -0.062
position_volatility           -0.041
ctr                           -0.018
ga4_<NA>                       0.005
dtype: float64

Confusion matrix (rows = actual, cols = predicted), [[TN, FP], [FN, TP]]:
[[28212   462]
 [11786  1118]]

              precision    recall  f1-score   support

           0      0.705     0.984     0.822     28674
           1      0.708     0.087     0.154     12904

    accuracy                          0.705     41578
   macro avg      0.706     0.535     0.488     41578
weighted avg      0.706     0.705     0.615     41578


False negatives: 11,786  |  False positives: 462

Mean honest-feature va

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.